# Week 5 — Customer Segmentation with K-Means

**Theme:** Unsupervised learning II — clustering (k-means)

A mall wants to understand its customers so it can target promotions better.
Nobody has labeled customers as "budget shopper" or "big spender" — but if we
plot income vs. spending, natural groups might just... appear. That's what
**clustering** does: find groups of similar points with no labels given.

**K-Means algorithm, in one paragraph:** pick `k` random cluster centers ->
assign every point to its nearest center -> move each center to the average of
its assigned points -> repeat until centers stop moving.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 1. Create a synthetic customer dataset

We simulate 5 realistic customer segments (this also means we secretly *know*
the "true" answer, which is useful for checking whether K-Means finds it).

In [ ]:
rng = np.random.default_rng(42)

segments = [
    # (mean_income_k$, mean_spending_score, n_customers)
    (25, 20, 40),   # low income, low spending
    (25, 80, 40),   # low income, high spending (impulsive)
    (55, 50, 40),   # mid income, mid spending
    (85, 20, 40),   # high income, low spending (frugal)
    (85, 85, 40),   # high income, high spending
]

incomes, spending = [], []
for mean_income, mean_spend, n in segments:
    incomes.append(rng.normal(mean_income, 5, n))
    spending.append(rng.normal(mean_spend, 8, n))

income = np.concatenate(incomes)
spending_score = np.concatenate(spending)
X = np.column_stack([income, spending_score])
print("Customers:", X.shape[0])

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6, edgecolor="k")
plt.title("Customers: Annual Income vs. Spending Score (unlabeled)")
plt.xlabel("Annual income ($k)")
plt.ylabel("Spending score (1-100)")
plt.show()

## 2. How many clusters? The elbow method

We don't know `k` in advance. We try several values and plot "inertia" (how
tightly packed each cluster is) — look for the "elbow" where adding more
clusters stops helping much.

In [ ]:
inertias = []
k_range = range(1, 10)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Silhouette score is another way to pick k: higher is better (max 1.0)
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, km.labels_)
    print(f"k={k}: silhouette score = {score:.3f}")

## 3. Run K-Means with the chosen k

Both the elbow plot and the silhouette scores should point toward **k=5** —
which matches how we generated the data.

In [ ]:
k = 5
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap="tab10", alpha=0.7, edgecolor="k")
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            c="red", marker="X", s=250, edgecolor="black", label="cluster center")
plt.title(f"K-Means Clustering (k={k})")
plt.xlabel("Annual income ($k)")
plt.ylabel("Spending score (1-100)")
plt.legend()
plt.show()

## 4. Interpret the segments

Turn the raw cluster centers into a business-readable table.

In [ ]:
import pandas as pd

summary = pd.DataFrame(kmeans.cluster_centers_, columns=["avg_income_k$", "avg_spending_score"])
summary["n_customers"] = pd.Series(cluster_labels).value_counts().sort_index().values
summary.index.name = "cluster"
summary

## 5. 초기 centroid가 결과를 바꾼다

K-Means는 "임의로 고른 초기 centroid"에서 출발해서 반복적으로 개선해 나가는
알고리즘입니다. 그런데 이 **출발점이 나쁘면**, 알고리즘이 수렴하긴 해도
전혀 다른(더 나쁜) 결과로 수렴할 수 있습니다 — 데이터나 `k`가 바뀐 게
아니라, 순전히 "어디서 출발했는가"만 바뀐 겁니다.

아래에서 나쁜 초기 centroid와 좋은 초기 centroid를 각각 손으로 골라서,
K-Means가 몇 번 반복(iteration) 만에 어떻게 다르게 수렴하는지 직접 눈으로
비교해봅니다.

In [ ]:
# 서로 뚜렷하게 구분되는 3개의 진짜 그룹을 만듭니다:
# 위쪽에 크고 촘촘한 그룹 하나, 아래 왼쪽/오른쪽에 작은 그룹 두 개.
rng_init = np.random.default_rng(7)

def make_blob(center, n, std):
    return rng_init.normal(center, std, size=(n, 2))

X_top = make_blob((0, 2), 90, 0.35)
X_left = make_blob((-1, 0.1), 25, 0.25)
X_right = make_blob((1, 0.1), 25, 0.25)
X_demo = np.vstack([X_top, X_left, X_right])

plt.figure(figsize=(5, 4))
plt.scatter(X_demo[:, 0], X_demo[:, 1], alpha=0.6, edgecolor="k")
plt.title("3 true groups (unlabeled) -- top group is bigger & denser")
plt.show()

#### Lloyd's algorithm을 한 단계씩 직접 실행

K-Means 내부에서 실제로 벌어지는 두 단계 — **(1) 가장 가까운 centroid에
할당 → (2) 각 그룹의 평균으로 centroid 이동** — 를 `scikit-learn` 없이
직접 구현해서, 초기 centroid를 다르게 줬을 때 매 iteration마다 그림이
어떻게 달라지는지 비교해봅니다.

In [ ]:
def assign_to_nearest(X, centers):
    dist2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return dist2.argmin(axis=1)

def move_to_mean(X, labels, centers, k):
    new_centers = centers.copy()
    for j in range(k):
        pts = X[labels == j]
        if len(pts) > 0:
            new_centers[j] = pts.mean(axis=0)
    return new_centers

def run_lloyd_and_plot(X, init_centers, title):
    centers = init_centers.copy()
    colors = ["tab:red", "tab:blue", "tab:green"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(title, fontsize=14)
    for it in range(1, 7):
        labels = assign_to_nearest(X, centers)
        ax = axes.flat[it - 1]
        for j in range(3):
            pts = X[labels == j]
            ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=25, alpha=0.7)
        ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="+", s=250, linewidths=3)
        ax.set_title(f"Iteration {it}")
        centers = move_to_mean(X, labels, centers, 3)
    plt.tight_layout()
    plt.show()

#### 나쁜 초기 centroid

세 개의 초기 centroid 중 **두 개를 위쪽의 크고 촘촘한 그룹 안에** 몰아넣고,
나머지 하나는 위쪽 그룹과 오른쪽 그룹 사이 애매한 위치에 둡니다. 아래
왼쪽/오른쪽 그룹 근처에는 초기 centroid가 하나도 없다는 점에 주목하세요.

In [ ]:
bad_init = np.array([
    [0.05, 2.7],   # 위쪽 그룹 안
    [0.30, 2.15],  # 역시 위쪽 그룹 안 (첫 번째와 아주 가까움)
    [0.75, 1.0],   # 위쪽 그룹과 오른쪽 그룹 사이, 애매한 위치
])
run_lloyd_and_plot(X_demo, bad_init, "Bad initial centroids")

**무슨 일이 일어났나요?** 처음 두 centroid가 위쪽 그룹 안에서 시작했기
때문에, 원래 하나였던 위쪽 그룹이 빨강/파랑 두 개로 억지로 쪼개졌습니다.
반면 세 번째 centroid는 아래로 끌려 내려가면서, 원래 서로 멀리 떨어진
왼쪽/오른쪽 두 그룹을 초록색 하나로 묶어버렸습니다. `k=3`은 맞았지만,
**출발점이 나빴던 탓에 "진짜" 그룹 구조와는 다른 결과로 수렴**한 겁니다.

#### 좋은 초기 centroid

이번엔 세 개의 초기 centroid를 **세 그룹에 하나씩** 골고루 떨어뜨려
놓습니다 — 정확한 중심일 필요는 없고, 각 그룹 영역 안에만 있으면 됩니다.

In [ ]:
good_init = np.array([
    [0.2, 1.6],   # 위쪽 그룹 안
    [-0.8, 0.4],  # 왼쪽 그룹 안
    [0.8, -0.3],  # 오른쪽 그룹 안
])
run_lloyd_and_plot(X_demo, good_init, "Good initial centroids")

**차이가 보이시나요?** 좋은 초기 centroid는 Iteration 1부터 이미 "진짜"
3개 그룹과 거의 일치하고, 그 뒤로는 centroid가 거의 움직이지 않은 채 바로
수렴합니다. 나쁜 초기화처럼 그룹이 잘못 쪼개지거나 합쳐지는 일이 없습니다
— 데이터와 `k`는 완전히 같은데, **오직 시작점만 다릅니다.**

#### scikit-learn 기본값과 비교하기

`scikit-learn`의 `KMeans`는 기본값으로 `init='k-means++'`(무작위로
아무 데나 찍는 대신, 서로 멀리 떨어지도록 영리하게 초기 centroid를 고름)와
`n_init=10`(서로 다른 초기화로 10번 실행해서 inertia가 가장 낮은 결과를
선택)을 함께 사용합니다. 왜 이렇게 하는지 숫자로 확인해봅니다.

In [ ]:
km_bad = KMeans(n_clusters=3, init=bad_init, n_init=1, random_state=0)
km_bad.fit(X_demo)

km_good = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=42)
km_good.fit(X_demo)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, km, title in [
    (axes[0], km_bad, f"Bad initial centroids\n(inertia={km_bad.inertia_:.1f})"),
    (axes[1], km_good, f"k-means++, n_init=10 (default)\n(inertia={km_good.inertia_:.1f})"),
]:
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=km.labels_, cmap="tab10", s=25, alpha=0.7, edgecolor="k", linewidth=0.3)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c="black", marker="X", s=200, edgecolor="white")
    ax.set_title(title)
plt.tight_layout()
plt.show()

print(f"Bad init inertia:  {km_bad.inertia_:.1f}")
print(f"Good init inertia: {km_good.inertia_:.1f}  (낮을수록 좋음)")

나쁜 초기화는 inertia(각 점과 자기 centroid 사이 거리 제곱의 합)가 훨씬
높습니다 — 같은 `k`, 같은 데이터인데도 **더 나쁜 클러스터링**으로
수렴했다는 뜻입니다. `n_init=10`으로 여러 번 시도하면, 운 나쁘게 나쁜
초기화 하나를 뽑더라도 그보다 나은 다른 시도가 선택되기 때문에 이런 문제를
훨씬 잘 피할 수 있습니다. (완전히 사라지는 건 아닙니다 — 데이터가 아주
애매하게 겹쳐 있으면 `n_init`을 늘려도 나쁜 결과가 나올 수 있습니다.)

## Try it yourself

1. **Pick the wrong k.** Re-run K-Means with `k=2` and `k=8` and re-plot — how
   does the clustering change? Which one looks "wrong" given the elbow plot?
2. **Name the segments.** Based on the summary table, write a one-word label
   for each cluster (e.g. "frugal high earners", "impulsive low earners").
3. **Add a third feature.** Simulate a `visits_per_month` column and re-run
   K-Means on all 3 features — plot 2 of the 3 dimensions to visualize.
4. **Compare to Week 4.** Both PCA and K-Means are unsupervised — what's the
   key difference in what each one is *for*? (Hint: one compresses features,
   the other groups examples.)

---
## 🎯 캡스톤: 스터디 그룹 매칭 서비스

가상의 학생 120명의 "선호 공부 시작 시각 / 선호 그룹 인원 / 몰입 강도" 더미 데이터를 드립니다. 위에서 배운 K-Means로 이들을 몇 개의 "스터디 성향 그룹"으로 나눠보고, **여러분 자신의 선호도**를 입력해서 어떤 그룹에 가장 잘 맞는지 찾아보세요.

**확장 아이디어:** 실제 스터디 모집 설문(선호 시간대, 인원, 강도 등)을 만들어 친구들 응답을 모으면, 이 코드로 진짜 "스터디 그룹 자동 매칭"을 만들 수 있습니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(21)

# 4가지 스터디 성향 (선호 시작 시각 0-23시, 선호 인원 1-8명, 몰입 강도 1-10)
archetypes = {
    "새벽 집중파":      (6, 2, 9),
    "카공족(카페 공부)": (14, 4, 5),
    "왁자지껄 그룹형":   (16, 7, 3),
    "밤샘 벼락치기형":   (23, 1, 8),
}

rows, true_type = [], []
for name, (hour, size, intensity) in archetypes.items():
    n = 30
    hours = np.clip(rng.normal(hour, 2, n), 0, 23)
    sizes = np.clip(rng.normal(size, 1.2, n), 1, 8)
    intensities = np.clip(rng.normal(intensity, 1.5, n), 1, 10)
    rows.append(np.column_stack([hours, sizes, intensities]))
    true_type += [name] * n

group_prefs_df = pd.DataFrame(np.vstack(rows), columns=["preferred_hour", "group_size_pref", "focus_intensity"])
group_prefs_df["true_type"] = true_type  # 실제 서비스라면 이런 라벨은 없습니다 -- 확인용으로만 사용
group_prefs_df.head()

### 여러분의 과제

1. `group_prefs_df[["preferred_hour", "group_size_pref", "focus_intensity"]]`에 `KMeans(n_clusters=4)`를 학습시키세요. (원한다면 Week 5 본문처럼 elbow method로 먼저 k를 확인해봐도 좋습니다.)
2. 각 클러스터의 평균 `preferred_hour` / `group_size_pref` / `focus_intensity`를 표로 출력해서, 클러스터마다 어떤 "스터디 성향"인지 이름을 붙여보세요.
3. 아래 `my_prefs`에 여러분 자신의 선호도를 입력하고, 학습된 `kmeans.predict()`로 어느 클러스터에 속하는지 확인하세요.
4. 여러분과 같은 클러스터에 속한 가상 학생이 몇 명인지, 그리고 (선택) 3개 특성 중 2개를 골라 산점도에 여러분의 위치를 겹쳐 그려보세요.

In [ ]:
# TODO 1: KMeans(n_clusters=4)를 학습시키세요.


# TODO 2: 각 클러스터의 평균 특성을 표로 출력하고, 클러스터별로 이름을 붙여보세요.


# TODO 3: 나의 선호도를 입력하고 어느 클러스터에 속하는지 예측해보세요.
my_prefs = {
    "preferred_hour": None,     # 예: 20 (밤 8시)
    "group_size_pref": None,    # 예: 3
    "focus_intensity": None,    # 예: 7
}

# TODO 4: 같은 클러스터의 학생 수를 세어보고, 산점도에 나의 위치를 겹쳐 그려보세요.

---
## 🍿 확장 캡스톤 (선택): 영화 취향 클러스터링 & 추천

앞의 캡스톤이 재밌었다면 한 단계 더 도전해봅시다. 실제
[MovieLens](https://movielens.org/) 평점 데이터로 **"취향이 비슷한
사용자를 K-Means로 묶고, 같은 그룹 사람들이 높게 준 영화를 추천"**하는
간단한 추천 시스템을 만들어봅니다.

**핵심 아이디어**

1. 다양한 장르에서 **대표 영화(reference movies)** 20~30개를 고른다.
2. 각 사용자를 "대표 영화들에 준 평점" 벡터로 표현한다 — `[5, 4, 2, 5, ...]`.
3. 이 벡터에 K-Means를 적용해 사용자를 몇 개의 "취향 그룹(cluster)"으로 나눈다.
4. 새로운 사용자가 대표 영화 중 일부만 평가해도, 어느 그룹에 속하는지
   예측하고 **같은 그룹 사람들이 (아직 안 본) 다른 영화에 준 평균 평점**으로
   추천 목록을 만든다.

**결측치 문제:** 실제로는 한 사람이 모든 대표 영화를 평가하지 않았을 수
있습니다. 여기서는 두 가지 서로 다른 상황에서 각각 **평균값으로 채우는**
방법을 씁니다 — (a) 클러스터링용 벡터에 빈칸이 있으면 **그 사용자 자신의
평균 평점**으로, (b) 추천 대상 영화에 평점이 부족하면 **그 클러스터의
평균 평점**으로.

### 1. MovieLens 데이터 불러오기

[MovieLens ml-latest-small](https://files.grouplens.org/datasets/movielens/ml-latest-small.zip)
데이터셋을 씁니다 — 사용자 약 600명이 영화 약 9,700편에 남긴 평점 약
100,000개. Kaggle과 달리 **로그인이나 계정 없이** 그냥 다운로드할 수 있는
공개 파일입니다.

In [ ]:
import urllib.request
import zipfile
import io

url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
with urllib.request.urlopen(url) as resp:
    with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
        z.extractall(".")

ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")
print("ratings:", ratings.shape, " movies:", movies.shape)
ratings.head()

### 2. 대표 영화(reference movies) 고르기

특정 장르에 치우치지 않도록, **장르별로 개수를 정해서** 그 안에서 평점을
많이 받은(=많은 사람이 본) 영화 위주로 고릅니다. 평점 수가 너무 적은
영화는 "대표성"이 없으니 제외합니다.

In [ ]:
genre_targets = {
    "Action": 5, "Comedy": 5, "Drama": 5, "Sci-Fi": 5,
    "Animation": 3, "Romance": 3, "Horror": 2,
}
POPULAR_MIN = 50  # 이 정도는 평가받아야 "많은 사람이 아는 영화"로 침

rating_counts = ratings.groupby("movieId").size().rename("n_ratings")
movies_stats = movies.merge(rating_counts, on="movieId")
popular = movies_stats[movies_stats["n_ratings"] >= POPULAR_MIN]

reference_ids = []
for genre, n in genre_targets.items():
    candidates = popular[popular["genres"].str.contains(genre)].sort_values("n_ratings", ascending=False)
    reference_ids.extend(candidates["movieId"].head(n).tolist())
reference_ids = list(dict.fromkeys(reference_ids))  # 여러 장르에 겹쳐 뽑힌 경우 중복 제거

reference_movies = movies_stats[movies_stats["movieId"].isin(reference_ids)].reset_index(drop=True)
print("대표 영화 개수:", len(reference_movies))
reference_movies[["title", "genres", "n_ratings"]]

### 3. 사용자 x 대표영화 평점 행렬 만들기

대표 영화 중 **일정 비율 이상 평가한 사용자만** 학습에 씁니다 (너무 많은
빈칸을 평균으로 채우면 그 사용자의 진짜 취향이 흐려지기 때문). 남은 빈칸은
**그 사용자 자신의 평균 평점**으로 채웁니다.

`ELIGIBLE_FRAC`으로 채택 기준을 조절할 수 있습니다 — 값을 낮추면 더 많은
사용자가 남지만, 각 사용자 벡터에 채워야 할 빈칸도 늘어납니다.

In [ ]:
ELIGIBLE_FRAC = 0.5  # 대표 영화 중 이 비율 이상 평가한 사용자만 사용

ref_ratings = ratings[ratings["movieId"].isin(reference_movies["movieId"])]
n_ref = len(reference_movies)
user_ref_counts = ref_ratings.groupby("userId")["movieId"].nunique()
eligible_users = user_ref_counts[user_ref_counts >= ELIGIBLE_FRAC * n_ref].index
n_total_users = ratings["userId"].nunique()
print(f"학습에 쓸 사용자: {len(eligible_users)} / {n_total_users}")

pivot = ref_ratings[ref_ratings["userId"].isin(eligible_users)].pivot_table(
    index="userId", columns="movieId", values="rating"
)
pivot = pivot.reindex(columns=reference_movies["movieId"].values)  # 대표 영화 순서 고정

user_means = pivot.mean(axis=1)
pivot_filled = pivot.apply(lambda row: row.fillna(user_means[row.name]), axis=1)
print("빈칸 개수 (채우기 전 -> 후):", pivot.isna().sum().sum(), "->", pivot_filled.isna().sum().sum())
pivot_filled.shape

### 4. K-Means로 취향 그룹 나누기

모든 특징(대표 영화 각각의 평점)이 이미 같은 0.5~5.0 척도라서, 이번에는
Part B의 Ridge/Lasso 때와 달리 `StandardScaler`가 필요 없습니다.

In [ ]:
N_CLUSTERS = 4  # 몇 개의 "취향 그룹"으로 나눌지 -- 정답은 없으니 바꿔서 실험해봐도 좋습니다

kmeans_movie = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
user_cluster = pd.Series(
    kmeans_movie.fit_predict(pivot_filled.values), index=pivot_filled.index, name="cluster"
)
print("클러스터별 사용자 수:")
print(user_cluster.value_counts().sort_index())

### 5. 클러스터별 영화 평균 평점 (추천용)

이제 **대표 영화가 아닌 모든 영화**에 대해, 각 클러스터 사용자들이 매긴
평균 평점을 계산합니다. 한두 명만 평가한 (cluster, movie) 조합은 평균이
믿을 만하지 않으니 `MIN_CLUSTER_RATINGS`개 미만이면 제외합니다.

In [ ]:
MIN_CLUSTER_RATINGS = 3

ratings_labeled = ratings[ratings["userId"].isin(eligible_users)].merge(
    user_cluster, left_on="userId", right_index=True
)
cluster_movie_stats = (
    ratings_labeled.groupby(["cluster", "movieId"])["rating"]
    .agg(["mean", "count"])
    .reset_index()
)
cluster_movie_stats = cluster_movie_stats[cluster_movie_stats["count"] >= MIN_CLUSTER_RATINGS]
print("믿을 만한 (cluster, movie) 조합 수:", len(cluster_movie_stats))

# 클러스터마다 가장 높게 평가받은 영화 5개씩 미리보기
for c in sorted(user_cluster.unique()):
    top5 = (
        cluster_movie_stats[cluster_movie_stats["cluster"] == c]
        .merge(movies, on="movieId")
        .sort_values("mean", ascending=False)
        .head(5)
    )
    print(f"\n--- Cluster {c} 최고 평점 영화 Top 5 ---")
    print(top5[["title", "mean", "count"]].to_string(index=False))

### 여러분의 과제

1. 위 `reference_movies`의 `title` 목록을 보고, `my_ratings_list`에서
   **본 적 있는 영화의 인덱스에 1~5점**을 채워 넣으세요 (안 본 영화는
   `None`으로 남겨둡니다). 최소 3~4개는 채워야 취향이 의미 있게 반영됩니다.
2. `None`으로 남은 자리를 **내가 매긴 평점들의 평균**으로 채워서 완전한
   벡터를 만드세요.
3. `kmeans_movie.predict()`로 내가 어느 클러스터에 속하는지 예측하세요.
4. `cluster_movie_stats`에서 내 클러스터의, **아직 평가하지 않은(대표
   영화가 아닌)** 영화 중 평균 평점이 가장 높은 Top-5를 출력하세요.

In [ ]:
print(reference_movies[["title"]].reset_index(drop=True))

# TODO 1: 본 영화의 인덱스에 1~5점을 채우세요. (예: my_ratings_list[0] = 5)
my_ratings_list = [None] * len(reference_movies)


# TODO 2: None을 내가 매긴 평점들의 평균으로 채워서 my_vector를 완성하세요.
my_vector = None


# TODO 3: kmeans_movie.predict()로 내 클러스터를 예측하세요.
my_cluster = None


# TODO 4: 내 클러스터에서, 대표 영화가 아닌 영화 중 평균 평점 Top-5를 출력하세요.
